In [1]:
from pathlib import Path
import pandas as pd
import re

In [2]:
# VERSION SETTING
version_dict = {
    "original" : ["kd_tree", "angular", "octree"],
    "modified" : ["01_ring", "02_xyz", "golden_ref"]
}

selected_type = "modified"
version = version_dict[selected_type][2]

config_label = "26-03-05_delta"
version = f"{version}_{config_label}"
print(f"Selected version: {version}")

Selected version: golden_ref_26-03-05_delta


In [3]:
def parse(file_path):
    rows = []
    path_obj = Path(file_path)
    filename = path_obj.stem # Without suffix

    # 1. Parse the scene name and frame index from the file name.
    # scene-0000_00.log
    name_match = re.match(r"scene-(\d+)_(\d+)", filename)
    if name_match:
        scene_name = f"scene-{name_match.group(1)}"
        frame_index = int(name_match.group(2))
    else:
        scene_name = "unknown"
        frame_index = -1
    
    # 2. Patterns inside the log file.
    patterns = {
        'slice_number': r"Slice number:\s*(\d+)",
        'positions_bitstream_size_bytes': r"positions bitstream size\s*(\d+)\s*B",
        'positions_bitstream_size_bpp': r"positions bitstream size.*?\((\d+\.\d+)\s*bpp\)",
        'positions_processing_time_user': r"positions processing time \(user\):\s*(\d+\.\d+)\s*s",
        'total_frame_size_bytes': r"Total frame size\s*(\d+)\s*B",
        'total_bitstream_size_bytes': r"Total bitstream size\s*(\d+)\s*B",
        'processing_time_wall': r"Processing time \(wall\):\s*(\d+\.\d+)\s*s",
        'processing_time_user': r"Processing time \(user\):\s*(\d+\.\d+)\s*s",
    }

    try:
        with open(file_path, 'r') as f:
            content = f.read()
        
        # 1. Find the index of "Slice number:"
        slice_starts = [m.start() for m in re.finditer(r"Slice number:", content)]

        # 2. Iterate through the slice blocks
        for i, start_idx in enumerate(slice_starts):
            #. From the start index to the end index until next slice block starts
            end_idx = slice_starts[i+1] if i + 1 < len(slice_starts) else len(content)
            block = content[start_idx:end_idx]

            row = {
                'scene_name': scene_name,
                'frame_index': frame_index
            }

            for key, pattern in patterns.items():
                match = re.search(pattern, block)
                if match:
                    val = match.group(1)
                    if '.' in val:
                        row[key] = float(val)
                    else:
                        row[key] = int(val)
            
            rows.append(row)

    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return []
    
    return rows

In [4]:
experiment_dir = Path("/home/swnh/pgc/experiments")
csv_dir = experiment_dir / "csv"
csv_dir.mkdir(exist_ok=True)

scene_dirs = experiment_dir / version

print(f"Search logs for each scenes in: {scene_dirs}")
print(f"Save result to csv in: {csv_dir}")

Search logs for each scenes in: /home/swnh/pgc/experiments/golden_ref_26-03-05_delta
Save result to csv in: /home/swnh/pgc/experiments/csv


In [5]:
all_data = []
parse_count = 0

for scene_dir in scene_dirs.iterdir():
    print(f"[Start searching {scene_dir}]")
    if scene_dir.is_dir():
        log_files = sorted(scene_dir.glob("*.log"))
        
        for log_file in log_files:
            file_data = parse(log_file)
            parse_count += 1
            print(f"    >> Finished parsing {log_file}")
            if file_data:
                all_data.extend(file_data)
        print("\n")

df_all_data = pd.DataFrame(all_data)

if not df_all_data.empty:
    df_all_data = df_all_data.sort_values(by=['scene_name', 'frame_index']).reset_index(drop=True)
    df_all_data = df_all_data.round(3)

print(f"[Finished parsing {parse_count} files]")



[Start searching /home/swnh/pgc/experiments/golden_ref_26-03-05_delta/scene-0553]
    >> Finished parsing /home/swnh/pgc/experiments/golden_ref_26-03-05_delta/scene-0553/scene-0553_00.log
    >> Finished parsing /home/swnh/pgc/experiments/golden_ref_26-03-05_delta/scene-0553/scene-0553_01.log
    >> Finished parsing /home/swnh/pgc/experiments/golden_ref_26-03-05_delta/scene-0553/scene-0553_02.log
    >> Finished parsing /home/swnh/pgc/experiments/golden_ref_26-03-05_delta/scene-0553/scene-0553_03.log
    >> Finished parsing /home/swnh/pgc/experiments/golden_ref_26-03-05_delta/scene-0553/scene-0553_04.log
    >> Finished parsing /home/swnh/pgc/experiments/golden_ref_26-03-05_delta/scene-0553/scene-0553_05.log
    >> Finished parsing /home/swnh/pgc/experiments/golden_ref_26-03-05_delta/scene-0553/scene-0553_06.log
    >> Finished parsing /home/swnh/pgc/experiments/golden_ref_26-03-05_delta/scene-0553/scene-0553_07.log
    >> Finished parsing /home/swnh/pgc/experiments/golden_ref_26-03-05

In [7]:
file_name = f"all_data_{version}.csv"
csv_path = csv_dir / file_name

df_all_data.to_csv(csv_path, index=False)

print(f"Successfully saved to: {csv_path}")

Successfully saved to: /home/swnh/pgc/experiments/csv/all_data_golden_ref_26-03-05_delta.csv


In [6]:
import matplotlib.pyplot as plt

df_scene_avg = (
    df_all_data.groupby(['scene_name'])
    .mean(numeric_only=True)
    .drop(columns=['frame_index', 'slice_number'])
    .reset_index()
    .round(3)
)
df_scene_avg

,scene_name,positions_bitstream_size_bytes,positions_bitstream_size_bpp,positions_processing_time_user,total_frame_size_bytes,total_bitstream_size_bytes,processing_time_wall,processing_time_user
0,scene-0061,52425.949,12.079,0.025,52504.949,52504.949,0.083,0.037
1,scene-0103,45087.650,10.388,0.024,45166.650,45166.650,0.082,0.036
2,scene-0553,41484.561,9.561,0.023,41563.561,41563.561,0.083,0.035
3,scene-0655,43659.220,10.063,0.024,43738.220,43738.220,0.084,0.036
4,scene-0757,46571.415,10.731,0.024,46650.415,46650.415,0.083,0.036
5,scene-0796,55349.625,12.751,0.026,55428.625,55428.625,0.084,0.037
6,scene-0916,56021.927,12.908,0.026,56100.878,56100.878,0.085,0.038
7,scene-1077,54045.415,12.452,0.026,54124.415,54124.415,0.084,0.038
8,scene-1094,57836.800,13.329,0.026,57915.800,57915.800,0.085,0.038
9,scene-1100,57449.375,13.240,0.026,57528.375,57528.375,0.086,0.038
